In [ ]:
# hide
# no-output
import pathlib
import numpy as np
import pyquist as pq

# Load Freesound sounds from the copies cached with this book (no API key needed)
pq.paths.CACHE_DIR = pathlib.Path("../assets")
from pyquist.web import fetch_freesound

In [ ]:
def freesound(fsid: int, f_s: int = 44100) -> pq.Audio:
    """An instrument that plays the Freesound sound with ID `fsid`."""
    audio, _ = fetch_freesound(fsid)
    audio = audio.as_mono()
    audio.normalize()
    if audio.sample_rate != f_s:
        audio = audio.resample(f_s)         # all sounds in a score need one sample rate
    return audio


def ascii_to_score(ascii: str,
                   row_fsids: tuple[int, ...] = (399897, 46709, 673512)) -> pq.Score:
    """One row per drum, one column per step: an X plays that row's sound at that step."""
    rows = ascii.strip().splitlines()
    score = pq.Score()
    num_steps = max(len(row) for row in rows)
    for step in range(num_steps):
        for row, fsid in zip(rows, row_fsids):
            if len(row) > step and row[step] == "X":
                score.append((step / 4, {"fsid": fsid}))
    return score

In [ ]:
pattern = """
X-X-X-X-X-X-X-X-X-X-X-X-X-X-X-X-
----X-------X-------X-------X---
X-------X-------X-X-----X-------
"""                                           # rows: hi-hat, snare, kick

score = ascii_to_score(pattern)
for event in score[:8]:
    print(event)

# Each step is a sixteenth note: four steps per beat at 120 BPM
metronome = pq.BasicMetronome(bpm=120)
pq.play(score.render(freesound, metronome), normalize=True)